# AI 智能体框架的工程化
如何利用开源生态的顶级杀器 —— LangChain 与 LangGraph,装入企业级、标准化的框架体系中。

## 框架的 Tool Calling 抽象与标准化
1. 破除多模型适配痛点：理解为什么在企业级开发中，我们需要框架层（LangChain）来统一 OpenAI、Anthropic、Ollama 等不同厂商的 API 接口。
2. 掌握 `@tool` 装饰器的底层魔法：看透 LangChain 是如何利用 Python 类型提示（Type Hints）和 Pydantic，自动生成大模型所需的底层 JSON Schema 的。
3. 构建首个 LangGraph 预置智能体：放弃冗长的原生循环，使用 `create_react_agent` 一键拉起具备完整思考与工具调用能力的 Agent。

## 为什么需要 LangChain / LangGraph？
原生手写的痛点：
* 厂商协议不统一：OpenAI 的 Tool Calling 格式和 Anthropic (Claude) 或者国内的 Qwen、DeepSeek 格式存在微妙的差异。如果企业要求“一键切换底层大模型”，你手写的原生代码需要大量重构。
* 参数校验脆弱：大模型返回的参数是纯字符串（String），如果缺漏参数或类型错误，原生代码极易崩溃。
* 状态管理繁琐：手动维护 `messages` 列表、拼装 `tool_call_id` 和 `tool` 角色消息非常容易出错。

LangChain 的标准化解法 (Model I/O)：

LangChain 提供了一个统一的抽象层。你只需要写标准的 Python 函数，LangChain 会自动将其翻译成各大模型厂商认识的专有 JSON 格式；当模型返回调用请求时，框架又会自动将其解析并路由到对应的 Python 函数。

## 核心机制解析：`@tool` 与 `bind_tools`
在 LangChain 生态中，赋予大模型“手脚”只需要两步：

1. 定义工具 (The `@tool` Decorator)
    你只需要写一个带类型提示（Type Hints）和 Google 风格 Docstring 的 Python 函数，并加上 `@tool` 装饰器。LangChain 会在底层自动通过 Pydantic 提取这些信息，编译成大模型需要的 Schema。

In [ ]:
from langchain_core.tools import tool

@tool
def get_weather(city: str) -> str:
    """获取指定城市的天气状况。"""
    # 模拟外部 API 调用
    return f"{city}今天是晴天，气温 25 度。"

2. 绑定工具 (The `bind_tools` Method)

    所有的 LangChain 聊天模型（`ChatOpenAI`, `ChatAnthropic`, `ChatOllama` 等）都实现了一个标准方法 `bind_tools()`。它会在向模型发起请求时，悄悄把工具的 `JSON Schema` 塞进请求体中。


#### 一键拉起 LangGraph ReAct 智能体


In [ ]:
import os
from langchain_openai import ChatOpenAI
from langchain_core.tools import tool
from langgraph.prebuilt import create_react_agent

# ================= 1. 定义工具集 =================
@tool
def calculate_profit(revenue: float, cost: float) -> float:
    """计算企业的净利润。输入必须是总收入和总成本。"""
    return revenue - cost

@tool
def get_company_data(company_name: str) -> dict:
    """查询公司的财报数据。"""
    # 模拟数据库查询
    mock_db = {
        "TechCorp": {"revenue": 500000.0, "cost": 300000.0},
        "BioHealth": {"revenue": 800000.0, "cost": 750000.0}
    }
    return mock_db.get(company_name, {"error": "未找到该公司数据"})

tools = [calculate_profit, get_company_data]

# ================= 2. 初始化大模型 =================
# 你可以无缝将其替换为 ChatOllama, ChatAnthropic 等
llm = ChatOpenAI(model="gpt-4o", temperature=0)

# ================= 3. 构建 LangGraph 智能体 =================
# 这一行代码在底层构建了一个包含 "agent" 节点和 "tools" 节点的循环状态图 (StateGraph)
agent_executor = create_react_agent(llm, tools)

# ================= 4. 运行智能体 =================
query = "请帮我查询 TechCorp 公司的财报数据，并计算它的净利润是多少？"

print("🤖 智能体开始执行...\n")

# stream 方法允许我们观察 Agent 在图结构中每一步的流转状态
for step in agent_executor.stream({"messages": [("user", query)]}):
    for node_name, node_state in step.items():
        print(f"--- 触发节点: {node_name} ---")
        # 打印当前节点产生的最新消息
        latest_msg = node_state["messages"][-1]
        print(latest_msg.content if latest_msg.content else latest_msg.additional_kwargs)
        print("\n")


1. 底层思维映射：
   * 当你运行上面的实战代码时，你会看到控制台交替打印出 `agent` 节点和 `tools` 节点的信息。请回顾之前我们学习的 LangGraph 基础，在脑海中画出 `create_react_agent` 内部的有向有环图（DAG）结构。哪一个是普通边（Edge）？哪一个是条件边（Conditional Edge）？

2. 多参数复杂工具防御：
    * 在使用 `@tool` 装饰器时，如果你的函数参数是一个复杂的嵌套字典或列表（例如 `def create_user(user_info: dict):`），大模型经常会填错格式。在企业开发中，我们通常不会只用简单的 Type Hints，而是会传入一个显式的 Pydantic `BaseModel` 作为 args_schema。查阅 LangChain 官方文档，了解如何为 `@tool` 绑定一个强类型的 Pydantic Schema 以实现严格的输入校验。

## 全局状态管理（State Schema）与消息追加机制（Reducer）
让 Agent 在极其复杂的多步推理和多轮对话中，保证上下文数据无损、精准地在各个节点之间流转。
1. 解密 State Schema（全局状态）：理解有向有环图（DAG）中，数据是如何作为“状态机”在节点间传递的。
2. 掌握 Reducer（聚合器）的追加魔法：彻底搞懂 `Annotated` 与 `add_messages`，解决多轮对话历史被“覆盖清空”的致命 Bug。
3. 纯手工构建消息图（Message Graph）：不依赖预置 Agent，手写一个管理多轮对话状态的定制化 StateGraph。

#### 从“流水线”到“状态机”的思维跃迁
在传统的 LangChain LCEL（链式表达式）中，数据是单向流动的：Node A 的输出直接变成 Node B 的输入。一旦流过去，Node A 的原始数据往往就丢失了。

但在 LangGraph 中，我们构建的是一个状态机（State Machine）。
系统会维护一个全局的字典对象（State）。每一个节点（Node）在执行时，不仅能读取这个全局 State 的全部内容，在执行完毕后，还能通过返回一个字典来更新这个全局 State。

节点更新 State 的核心准则：
节点返回的字典，只会更新 State 中对应的键值（Key），不会影响其他不相关的 Key。

#### 覆盖 (Overwrite) vs 追加 (Append)
定义全局 State 通常使用 Python 的 `TypedDict`。在定义字段时，存在两种截然不同的更新策略：

| 状态更新策略               | 代码定义示例                                    | 行为表现                | 适用业务场景                        |
|----------------------|-------------------------------------------|---------------------|-------------------------------|
| 覆盖 (Overwrite)       | `current_status: str`                     | 节点返回的新值会直接覆盖旧值。     | 保存“当前处理进度”、“最新的摘要归纳”、“错误状态码”。 |
| 追加 (Append / Reduce) | `messages: Annotated[list, add_messages]` | 节点返回的新消息会拼接到历史列表末尾。 | 保存“多轮对话历史”、“工具调用的执行日志”。       |

深入解析 `Annotated` 与 `add_messages`：

在原生 Python 字典中，如果你更新 `{"messages": [新消息]}`，旧的 `messages` 列表会被直接替换，导致 Agent 失忆。

为了解决这个问题，LangGraph 引入了 Annotated 类型注解和聚合器函数（如官方提供的 `add_messages`，或者 Python 原生的 `operator.add`）。当图引擎看到这个标记时，它不会做“覆盖”替换，而是执行 `旧列表 + 新列表` 的合并操作。这正是维持多轮对话上下文的核心魔法！


#### 手工搭建多轮对话状态图
脱离 `create_react_agent`，纯手工用 LangGraph 的基本算子（StateGraph, Node, Edge）搭建一个能够记住用户多轮提问的对话机器人。

In [ ]:
from typing import Annotated, TypedDict
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage, BaseMessage
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages

# ================= 1. 定义全局状态 (State Schema) =================
# 使用 Annotated 和 add_messages 确保消息是追加而不是覆盖
class AgentState(TypedDict):
    messages: Annotated[list[BaseMessage], add_messages]
    # 你还可以在这里自由添加其他覆盖型状态，例如：
    # summary: str

# ================= 2. 初始化模型 =================
llm = ChatOpenAI(model="gpt-4o", temperature=0)

# ================= 3. 定义节点 (Nodes) =================
def chatbot_node(state: AgentState):
    """
    负责调用大模型的节点。
    它读取 state["messages"] 中的全部历史，传入大模型，
    并将大模型的新回复返回，用于追加到状态中。
    """
    print(f"\n[后台日志] 当前全局 State 包含 {len(state['messages'])} 条消息。")
    response = llm.invoke(state["messages"])

    # 返回的内容会通过 add_messages 自动追加到全局 messages 列表中
    return {"messages": [response]}

# ================= 4. 构建与编译图 (Build Graph) =================
# 将我们定义的 AgentState 传入构建器
builder = StateGraph(AgentState)

# 添加节点并命名
builder.add_node("chatbot", chatbot_node)

# 描绘流转边 (Edges)：START -> chatbot -> END
builder.add_edge(START, "chatbot")
builder.add_edge("chatbot", END)

# 编译成可执行应用
graph_app = builder.compile()

# ================= 5. 测试多轮对话状态保留 =================
print("🤖 启动手工状态图 (输入 'quit' 退出)")

# 我们需要在外层维护一个临时的 thread 状态字典来模拟连续对话
# (在后续课程中，我们将学习如何让框架自动持久化这些数据)
current_state = {"messages": []}

while True:
    user_input = input("\n🧑 用户: ")
    if user_input.lower() in ["quit", "exit"]:
        break

    # 将用户的新输入追加到当前的临时状态中
    current_state["messages"].append(HumanMessage(content=user_input))

    # 运行图
    final_state = graph_app.invoke(current_state)

    # 提取大模型的最新回复并打印
    ai_msg = final_state["messages"][-1]
    print(f"🤖 助手: {ai_msg.content}")

    # 将图更新后的完整状态同步给外层，准备下一轮
    current_state = final_state

1. 状态污染防御：在上面的循环测试中，如果我们不小心在 `chatbot_node` 节点里返回了 `{"messages": state["messages"] + [response]}`（即手动把历史拼接了一次又返回了），结合 `add_messages` 聚合器的特性，全局 State 里的历史记录会发生什么灾难性的变化？
2. 多模态状态扩展：假设你现在要为一个视觉 Agent 设计 `AgentState`，除了需要 `messages` 来保存对话历史，你还需要保存用户上传的“当前图片 URL”和“分析报告文本”。你将如何用 Python 的 `TypedDict` 定义这个包含了“追加更新”与“覆盖更新”混合逻辑的 State Schema？

## 条件路由（Conditional Edges）与动态循环控制
包含“思考 $\rightarrow$ 决定使用工具 $\rightarrow$ 观察结果 $\rightarrow$ 再次思考”完整闭环的 有向有环图（DAG）

1. 突破线性思维：理解为什么 Agent 需要“图中的环（Cycles）”，以及如何通过边（Edges）让数据在节点间反复流转。
2. 掌握条件边（Conditional Edges）：学会编写 Router 路由函数，让 Agent 根据大模型的输出动态决定“下一步该走哪条路”。
3. 纯手工组装 ReAct 智能体：不依赖 `create_react_agent` 预置黑盒，用最底层的 StateGraph API 一砖一瓦地搭建一个可无限循环纠错的 Agent。

#### 为什么我们需要“有向有环图（DAG）”？
传统的流水线（如数据预处理、常规的 RAG 链）都是单向的：`A -> B -> C -> 结束`。

但在真正的业务场景中，Agent 面对的任务是不确定的。比如用户问：“查一下微软的股价，然后用计算器算出它买 100 股多少钱？”
Agent 的执行逻辑必须是：
1. 调用 `get_stock` 工具（离开大脑）。
2. 必须能回到大脑，让大模型看一眼股票价格。
3. 调用 `calculator` 工具（再次离开大脑）。
4. 再次回到大脑，得出最终自然语言结论。

这种“折返跑”的逻辑，在 LangGraph 中被称为环（Cycle）。要实现环，我们不能预先写死所有的边，而是必须让大模型自己当“交通警察”——这就是 条件边（Conditional Edges） 的价值。


#### 路由函数（Router Function）的解剖
条件边的核心是一个普通的 Python 函数（即 Router）。

每次图引擎运行到一个分叉路口时，就会把当前的全局状态（State）喂给这个 Router 函数。
Router 检查状态中的最新信息，然后返回一个字符串指令（如 `"continue"` 或 `"end"`），图引擎再根据这个字符串，决定数据流向哪个节点。

在 ReAct 架构中，Router 的核心判断逻辑极其简单：
查看大模型吐出的最新一条消息。如果它带有 `tool_calls`（意味着它想调用工具），就去工具节点；如果没有，说明它已经得出结论了，走向结束。


手写一个包含动态路由的完整智能体。为了让代码在工业界更具鲁棒性，我们将引入 LangGraph 官方提供的 ToolNode（一个专门负责批量执行 tool_calls 的极佳内置节点）。

In [ ]:
from typing import Annotated, Literal, TypedDict
from langchain_openai import ChatOpenAI
from langchain_core.messages import BaseMessage, HumanMessage
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from langgraph.prebuilt import ToolNode
from langchain_core.tools import tool

# ================= 1. 定义状态与工具 =================
class AgentState(TypedDict):
    messages: Annotated[list[BaseMessage], add_messages]

@tool
def get_weather(city: str) -> str:
    """获取指定城市的天气。"""
    if "北京" in city: return "北京今天是暴雨，气温 20 度。"
    return f"{city}天气晴朗，气温 25 度。"

tools = [get_weather]
# ToolNode 是一个预置节点，它会自动解析 state["messages"] 中的 tool_calls 并执行，
# 然后生成 role="tool" 的 ToolMessage 拼接到状态中。
tool_node = ToolNode(tools)

# ================= 2. 初始化大模型并绑定工具 =================
llm = ChatOpenAI(model="gpt-4o", temperature=0).bind_tools(tools)

# ================= 3. 定义普通节点 =================
def call_model(state: AgentState):
    """大脑节点：负责思考和决定"""
    response = llm.invoke(state["messages"])
    return {"messages": [response]}

# ================= 4. 定义条件路由函数 (核心!) =================
def should_continue(state: AgentState) -> Literal["continue", "end"]:
    """
    交通警察：决定图的下一步走向
    注意这里的返回类型提示 Literal，这是 LangGraph 强类型约束的最佳实践。
    """
    messages = state["messages"]
    last_message = messages[-1] # 获取模型的最新回复

    # 如果模型决定调用工具，导向 action 节点
    if last_message.tool_calls:
        return "continue"

    # 如果没有工具调用，说明模型想直接回复用户，导向 END
    return "end"

# ================= 5. 编织有向有环图 =================
workflow = StateGraph(AgentState)

# 添加节点
workflow.add_node("agent", call_model)
workflow.add_node("action", tool_node)

# 定义流转逻辑
workflow.add_edge(START, "agent")

# 添加条件边：从 agent 节点出发，经过 should_continue 裁判，决定去哪
workflow.add_conditional_edges(
    "agent",
    should_continue,
    {
        "continue": "action", # 如果返回 "continue"，走向 action 节点
        "end": END            # 如果返回 "end"，走向图的结束点
    }
)

# 【重点】闭环边：工具执行完后，必须无条件回到 agent 节点进行下一步思考！
workflow.add_edge("action", "agent")

# 编译成可执行应用
app = workflow.compile()

# ================= 6. 测试流转逻辑 =================
query = "请问北京和上海的天气分别怎样？请分别查询后再告诉我。"
print("🤖 启动手写 ReAct 引擎...\n")

for output in app.stream({"messages": [HumanMessage(content=query)]}):
    for node_name, node_state in output.items():
        print(f"--- 节点执行完毕: [{node_name}] ---")
        msg = node_state["messages"][-1]

        if node_name == "agent" and msg.tool_calls:
            print(f"🧠 大脑决定调用工具: {[t['name'] for t in msg.tool_calls]}")
        elif node_name == "action":
            print(f"🛠️ 工具执行结果: {msg.content}")
        elif node_name == "agent" and not msg.tool_calls:
            print(f"✅ 最终结论: {msg.content}")
        print("-" * 40)

1. 死循环的工程隐患：在上面的图结构中，`agent -> action -> agent -> action...` 构成了一个理论上可以无限循环的环。如果在极少数情况下，大模型陷入了幻觉，反复生成参数完全错误、导致执行一直失败的 `tool_calls`，这个图就会一直死循环空跑，疯狂燃烧你的 API Token。
    * 思考：查阅 LangGraph 文档，图引擎在编译时（`compile()`）提供了一个什么关键参数，可以直接从底层物理切断这种无限循环，防止系统崩溃？

2. 多节点路由扩展：假设你现在的系统里不仅仅有普通的 `tools`，还有一个专门用于搜索企业私有知识库的 `rag_retriever`。你希望大模型能够根据问题，决定是调用通用工具（去 `action` 节点），还是去查知识库（去 `rag` 节点），或者是直接结束（去 `END`）。
    * 思考：你将如何修改 `should_continue` 路由函数内部的判断逻辑，以及 `add_conditional_edges` 的字典映射表，来支持这种“三岔路口”的复杂路由？